# Set up

## Load libraries

In [1]:
%%time

import boto3
import numpy as np
import os
import pandas as pd
import sagemaker
from IPython import display as indy

CPU times: user 605 ms, sys: 120 ms, total: 725 ms
Wall time: 617 ms


## Declare constants

In [2]:
%%time

bucket_sr = 'gen-xii'
prefix_sr = 'data/match-file'

CPU times: user 4 µs, sys: 1 µs, total: 5 µs
Wall time: 7.15 µs


# Create join key in sent data

## Get sent files

In [3]:
%%time

s3_ct = boto3.client('s3')
keys_lt = [
    object_dt['Key'] for object_dt in s3_ct.list_objects(Bucket=bucket_sr, Prefix=prefix_sr)['Contents']
    if not object_dt['Key'].endswith('/')
]
sent_keys_lt = [key_sr for key_sr in keys_lt if 'sent' in key_sr]

print('Sent files:')
for index_it, sent_key_sr in enumerate(sent_keys_lt, 1):
    print(f'{index_it:<2} {sent_key_sr.split("/")[-1]}')

Sent files:
1  Application_10012018_01012020.csv
2  Application_10012018_01012020.txt
3  Data Dictionary.xlsx
4  Dataset Creation 20220329.R
5  Debt_10012018_01012020.csv
6  Debt_10012018_01012020.txt
7  Income_10012018_01012020.csv
8  Income_10012018_01012020.txt
9  LN_10012018_01012020.csv
10 LN_10012018_01012020.txt
11 TU_10012018_01012020.csv
12 TU_10012018_01012020.txt
CPU times: user 150 ms, sys: 8.73 ms, total: 159 ms
Wall time: 336 ms


## Read in sent application data

In [4]:
%%time

sent_application_key_sr = [
    sent_key_sr for sent_key_sr in sent_keys_lt if sent_key_sr.endswith('Application_10012018_01012020.csv')
][0]
print(f'Sent application key: {sent_application_key_sr}\n')

sent_application_df = pd.read_csv(f's3://{bucket_sr}/{sent_application_key_sr}')

sent_application_df.info()

Sent application key: data/match-file/sent-data/Application_10012018_01012020.csv



<timed exec>:6: DtypeWarning: Columns (6,30) have mixed types. Specify dtype option on import or set low_memory=False.


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 972297 entries, 0 to 972296
Data columns (total 80 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   UniqueID                        972297 non-null  int64  
 1   bigAccountId                    972297 non-null  int64  
 2   bigDebtorId                     972297 non-null  int64  
 3   bitDebtor                       972297 non-null  int64  
 4   strCity                         971866 non-null  object 
 5   strName                         971871 non-null  object 
 6   strZipCode                      971870 non-null  object 
 7   ApplicationDate                 972297 non-null  int64  
 8   bitApproved                     972297 non-null  int64  
 9   bitSystemDecline                690231 non-null  float64
 10  ApprovalDate                    240309 non-null  float64
 11  bitFunded                       972297 non-null  int64  
 12  FundedDate      

## Handle mixed types in zip

It appears as though...

* pandas reads in the CSV in chunks.
* Some of the chunks are all numeric but have missing values, so pandas casts them to float (since its int type does not accept NAs).
* However, other chunks are partially non-numeric, containing a hyphen, which is why the entire column is object rather than float. 
* When pandas puts these chunks together, it idenfies the mixed types, but leaves them to us to resolve.

In [5]:
%%time

def get_top_3_values_grouped_by_length(ss):
    return (
        ss.groupby(ss.str.len(), dropna=False)
        .apply(lambda x: x.value_counts(dropna=False).nlargest(3).index.tolist())
    )

print('Before:')
print(get_top_3_values_grouped_by_length(sent_application_df['strZipCode']))

def standardize_strings(ot):
    try:
        return str(int(ot))
    except:
        return str(ot)

sent_application_df['strZipCode'] = sent_application_df['strZipCode'].apply(standardize_strings)
    
print('\nAfter:')
print(get_top_3_values_grouped_by_length(sent_application_df['strZipCode']))

Before:
strZipCode
4.0                                   [6051]
5.0                    [20019, 60620, 20020]
6.0                         [466116, 611115]
10.0    [95901-8242, 20607-2806, 21222-4859]
NaN              [20020.0, 20019.0, 60620.0]
Name: strZipCode, dtype: object

After:
strZipCode
1                                      [0]
3                          [nan, 802, 791]
4                       [8021, 8081, 8618]
5                    [20019, 20020, 60620]
6                 [466116, 611115, 322226]
10    [95901-8242, 20607-2806, 21222-4859]
Name: strZipCode, dtype: object
CPU times: user 1.47 s, sys: 114 ms, total: 1.59 s
Wall time: 1.59 s


## Get current IDs

In [6]:
%%time

id_lt = sent_application_df.columns[:3].tolist()

id_lt

CPU times: user 48 µs, sys: 7 µs, total: 55 µs
Wall time: 59.6 µs


['UniqueID', 'bigAccountId', 'bigDebtorId']

## Get unique percent for all except IDs

In [7]:
%%time

single_unique_percent_ss = (
    (sent_application_df.drop(columns=id_lt).nunique() / sent_application_df.shape[0])
    .sort_values(ascending=False)
)

single_unique_percent_ss.to_frame('pct_unique').nlargest(10, 'pct_unique')

CPU times: user 1.95 s, sys: 288 ms, total: 2.24 s
Wall time: 2.24 s


,pct_unique
dtmStampCreation,0.776994
dtmDeclined,0.591534
dtmApproved,0.198653
fltApprovedPayment,0.044355
fltAmountFinanced,0.033283
dtmFunded,0.030427
PTI,0.030410
fltApprovedPriceWholesale,0.029757
fltTaxGrossReceipts,0.027355
fltAdvance,0.026651


## Get unique percent for doubles

In [8]:
%%time

double_unique_percent_lt = []
first_column_sr = single_unique_percent_ss.index[0]

for second_column_sr in single_unique_percent_ss.index[1:]:
    unique_percent_ft = (
        sent_application_df.groupby([first_column_sr, second_column_sr], dropna=False).ngroup().nunique() / 
        sent_application_df.shape[0]
    )
    double_unique_percent_lt.append((first_column_sr, second_column_sr, unique_percent_ft))

double_unique_percent_ss = pd.Series(
    [tmp_te[-1] for tmp_te in double_unique_percent_lt], 
    index=[tmp_te[:-1] for tmp_te in double_unique_percent_lt]
).sort_values(ascending=False)

double_unique_percent_ss.to_frame('pct_unique').nlargest(10, 'pct_unique')

CPU times: user 1min 42s, sys: 10.2 s, total: 1min 52s
Wall time: 1min 52s


,pct_unique
"(dtmStampCreation, bitDebtor)",0.978676
"(dtmStampCreation, strZipCode)",0.818755
"(dtmStampCreation, strCity)",0.815589
"(dtmStampCreation, strName)",0.799479
"(dtmStampCreation, DealerZip)",0.796836
"(dtmStampCreation, bigDealerID)",0.796681
"(dtmStampCreation, DealerCity)",0.796614
"(dtmStampCreation, DealerState)",0.795633
"(dtmStampCreation, fltApprovedDebtToIncome)",0.795611
"(dtmStampCreation, dtmDeclined)",0.795532


## Get unique percent for triples

In [9]:
%%time

def get_unique_percent(previous_unique_percent_ss):
    next_unique_percent_lt = []
    previous_best_lt = list(previous_unique_percent_ss.index[0])
    for next_sr in single_unique_percent_ss.index.difference(previous_best_lt):
        next_lt = previous_best_lt + [next_sr]
        unique_percent_ft = (
            sent_application_df.groupby(next_lt, dropna=False).ngroup().nunique() / 
            sent_application_df.shape[0]
        )
        next_unique_percent_lt.append(tuple(next_lt + [unique_percent_ft]))
    next_unique_percent_ss = pd.Series(
        [tmp_te[-1] for tmp_te in next_unique_percent_lt],
        index=[tmp_te[:-1] for tmp_te in next_unique_percent_lt]
    ).sort_values(ascending=False)
    return next_unique_percent_ss
        
triple_unique_percent_ss = get_unique_percent(double_unique_percent_ss)

triple_unique_percent_ss.to_frame('pct_unique').nlargest(10, 'pct_unique')

CPU times: user 1min 44s, sys: 10 s, total: 1min 54s
Wall time: 1min 54s


,pct_unique
"(dtmStampCreation, bitDebtor, strZipCode)",0.999935
"(dtmStampCreation, bitDebtor, strCity)",0.999855
"(dtmStampCreation, bitDebtor, bigDealerID)",0.999763
"(dtmStampCreation, bitDebtor, DealerZip)",0.999731
"(dtmStampCreation, bitDebtor, DealerCity)",0.999692
"(dtmStampCreation, bitDebtor, strName)",0.998804
"(dtmStampCreation, bitDebtor, DealerState)",0.998659
"(dtmStampCreation, bitDebtor, fltApprovedDebtToIncome)",0.998601
"(dtmStampCreation, bitDebtor, dtmDeclined)",0.998557
"(dtmStampCreation, bitDebtor, fltApprovedPayment)",0.996182


## Get unique percent for quadruples

In [10]:
%%time

quadruple_unique_percent_ss = get_unique_percent(triple_unique_percent_ss)

quadruple_unique_percent_ss.to_frame('pct_unique').nlargest(10, 'pct_unique')

CPU times: user 1min 48s, sys: 9.68 s, total: 1min 57s
Wall time: 1min 57s


,pct_unique
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined)",0.999988
"(dtmStampCreation, bitDebtor, strZipCode, fltApprovedDebtToIncome)",0.999950
"(dtmStampCreation, bitDebtor, strZipCode, VehicleMake)",0.999950
"(dtmStampCreation, bitDebtor, strZipCode, VehicleYear)",0.999950
"(dtmStampCreation, bitDebtor, strZipCode, VehiclePriceWholesale)",0.999949
"(dtmStampCreation, bitDebtor, strZipCode, VehicleModel)",0.999949
"(dtmStampCreation, bitDebtor, strZipCode, fltApprovedDownTotal)",0.999947
"(dtmStampCreation, bitDebtor, strZipCode, bitNew)",0.999947
"(dtmStampCreation, bitDebtor, strZipCode, fltApprovedPriceWholesale)",0.999944
"(dtmStampCreation, bitDebtor, strZipCode, bitDealerTrack)",0.999943


## Get unique percent for quintuples

In [11]:
%%time

quintuple_unique_percent_ss = get_unique_percent(quadruple_unique_percent_ss)

quintuple_unique_percent_ss.to_frame('pct_unique').nlargest(10, 'pct_unique')

CPU times: user 2min 49s, sys: 9.2 s, total: 2min 58s
Wall time: 2min 58s


,pct_unique
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, bitNew)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, bitDealerTrack)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, VehicleYear)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, VehicleModel)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, VehicleMake)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, VehiclePriceWholesale)",0.999994
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, dtmFunded)",0.999993
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, PTI)",0.999993
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, fltAdvance)",0.999993
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, bigMileage_odometer)",0.999993


## Get unique percent for sextuples

In [12]:
%%time

sextuples_unique_percent_ss = get_unique_percent(quintuple_unique_percent_ss)

sextuples_unique_percent_ss.to_frame('pct_unique').nlargest(10, 'pct_unique')

CPU times: user 2min 47s, sys: 9.09 s, total: 2min 56s
Wall time: 2min 56s


,pct_unique
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, bitNew, ApplicationDate)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, bitNew, ApplicationDayOfWeek)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, bitNew, fltInsuredDisabilityPremium)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, bitNew, fltInsuredDisabilityAmount)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, bitNew, fltGapInsurance)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, bitNew, fltDownCash)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, bitNew, fltDocumentFee)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, bitNew, fltApprovedServiceContract)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, bitNew, fltApprovedPriceWholesale)",0.999995
"(dtmStampCreation, bitDebtor, strZipCode, dtmDeclined, bitNew, fltApprovedPayment)",0.999995


## Create join key

In [13]:
%%time

join_lt = list(sextuples_unique_percent_ss.index[0])
join_sr = 'JoinKey'
sent_application_df[join_sr] = (
    sent_application_df
    .loc[:, join_lt]
    .astype(str)
    .apply(lambda x: '_'.join(x), axis=1)
)

sent_application_df[join_sr]

CPU times: user 6.64 s, sys: 156 ms, total: 6.8 s
Wall time: 6.81 s


0         2018-10-01T19:13:56Z_1_39110_2018-10-01T19:14:...
1         2018-10-01T19:13:56Z_0_39110_2018-10-01T19:14:...
2         2018-10-01T23:35:46Z_1_63136_2018-10-01T23:35:...
3         2018-10-01T23:35:46Z_0_63136_2018-10-01T23:35:...
4         2018-10-01T19:05:19Z_1_61610_2018-10-01T19:05:...
                                ...                        
972292    2019-04-11T00:37:58Z_0_61364_2019-04-11T00:38:...
972293        2019-04-10T23:00:23Z_1_87401_nan_0.0_20190410
972294    2019-04-10T22:01:27Z_1_78574_2019-04-10T22:01:...
972295    2019-04-10T23:48:10Z_1_98466_2019-04-10T23:48:...
972296    2019-04-11T00:58:31Z_1_87120_2019-04-11T15:01:...
Name: JoinKey, Length: 972297, dtype: object

## Drop duplicates

In [14]:
%%time

shape_te = sent_application_df.shape

print(f'Shape before: {shape_te}')

sent_application_df = sent_application_df.loc[:, id_lt + [join_sr]].drop_duplicates(join_sr)

shape_te2 = sent_application_df.shape

print(f'Shape after: {shape_te2}')
print(f'Difference: {np.array(shape_te2) - np.array(shape_te)}')

Shape before: (972297, 81)
Shape after: (972292, 4)
Difference: [ -5 -77]
CPU times: user 651 ms, sys: 172 ms, total: 823 ms
Wall time: 823 ms


# Create join key in return data

## Get return files

In [15]:
%%time

return_keys_lt = [key_sr for key_sr in keys_lt if 'return' in key_sr]

print('Return files:')
for index_it, return_key_sr in enumerate(return_keys_lt, 1):
    print(f'{index_it:<2} {return_key_sr.split("/")[-1]}')

Return files:
1  PRM.EDTOUT.DGMPRSTG.1273669.PERF.CSV
2  PRM.EDTOUT.DGMPRSTG.File2_SCRAM.csv
3  PRM.EDTOUT.DGMPRSTG.File3_SCRAM.csv
4  PRM.EDTOUT.DGMPRSTG.File4_SCRAM.csv
5  PRM.EDTOUT.DGMPRSTG.File5_SCRAM.csv
6  PRM.EDTOUT.DGMPRSTG.P467632.20180930-ACC2-P001.CSV
7  PRM.EDTOUT.DGMPRSTG.P467632.20180930.REJ2.P001.CSV
8  PRM.EDTOUT.DGMPRSTG.P467633.20181231-ACC2-P001.CSV
9  PRM.EDTOUT.DGMPRSTG.P467633.20181231.REJ2.P001.CSV
10 PRM.EDTOUT.DGMPRSTG.P467634.20190331-ACC2-P001.CSV
11 PRM.EDTOUT.DGMPRSTG.P467634.20190331.REJ2.P001.CSV
12 PRM.EDTOUT.DGMPRSTG.P467635.20190630-ACC2-P001.CSV
13 PRM.EDTOUT.DGMPRSTG.P467635.20190630.REJ2.P001.CSV
14 PRM.EDTOUT.DGMPRSTG.P467636.20190930-ACC2-P001.CSV
15 PRM.EDTOUT.DGMPRSTG.P467636.20190930.REJ2.P001.CSV
16 Prestige Data Dictionary - PERF.xlsx
CPU times: user 0 ns, sys: 502 µs, total: 502 µs
Wall time: 452 µs


## Get information about them

In [16]:
%%time

return_files_dt = {}

for key_sr in return_keys_lt:
    if key_sr.lower().endswith('.csv'):
        file_sr = key_sr.split("/")[-1]
        return_file_dt = {'key_sr': key_sr}
        try:
            read_csv_dt = dict(sep='|', encoding='ISO-8859-1') # Default utf-8 does not work for Scram3
            columns_ix = pd.read_csv(f's3://{bucket_sr}/{key_sr}', nrows=1, **read_csv_dt).columns
            ncols_it = len(columns_ix)
            if ncols_it == 1:
                columns_ix = pd.read_csv(f's3://{bucket_sr}/{key_sr}', nrows=1).columns
                ncols_it = len(columns_ix)
            nrows_it = pd.read_csv(f's3://{bucket_sr}/{key_sr}', usecols=[0], **read_csv_dt).shape[0]
            return_file_dt.update({
                'ncols_it': ncols_it,
                'nrows_it': nrows_it,
                'columns_ix': columns_ix
            })
        except Exception as en:
            return_file_dt.update({'exception': str(en)})
        return_files_dt.update({file_sr: return_file_dt})

return_files_df = pd.DataFrame(return_files_dt).T

return_files_df

CPU times: user 1min 43s, sys: 14.1 s, total: 1min 57s
Wall time: 9min 35s


,key_sr,ncols_it,nrows_it,columns_ix
PRM.EDTOUT.DGMPRSTG.1273669.PERF.CSV,data/match-file/return-data/PRM.EDTOUT.DGMPRST...,29,958287,"Index(['uniqueid', 'bigaccountid', 'bigdebtori..."
PRM.EDTOUT.DGMPRSTG.File2_SCRAM.csv,data/match-file/return-data/PRM.EDTOUT.DGMPRST...,193,972297,"Index(['UniqueID', 'bigAccountId', 'bigDebtorI..."
PRM.EDTOUT.DGMPRSTG.File3_SCRAM.csv,data/match-file/return-data/PRM.EDTOUT.DGMPRST...,63,1472267,"Index(['UniqueID', 'bigAccountId', 'bigDebtorI..."
PRM.EDTOUT.DGMPRSTG.File4_SCRAM.csv,data/match-file/return-data/PRM.EDTOUT.DGMPRST...,25,3559194,"Index(['UniqueID', 'bigAccountId', 'bigDebtorI..."
PRM.EDTOUT.DGMPRSTG.File5_SCRAM.csv,data/match-file/return-data/PRM.EDTOUT.DGMPRST...,80,972297,"Index(['UniqueID', 'bigAccountId', 'bigDebtorI..."
PRM.EDTOUT.DGMPRSTG.P467632.20180930-ACC2-P001.CSV,data/match-file/return-data/PRM.EDTOUT.DGMPRST...,1846,193964,"Index(['permId', 'creditAsOfDate_creditAsOfDat..."
PRM.EDTOUT.DGMPRSTG.P467632.20180930.REJ2.P001.CSV,data/match-file/return-data/PRM.EDTOUT.DGMPRST...,23,49,"Index(['customerInput_Uniqueid', 'customerInpu..."
PRM.EDTOUT.DGMPRSTG.P467633.20181231-ACC2-P001.CSV,data/match-file/return-data/PRM.EDTOUT.DGMPRST...,1846,208297,"Index(['permId', 'creditAsOfDate_creditAsOfDat..."
PRM.EDTOUT.DGMPRSTG.P467633.20181231.REJ2.P001.CSV,data/match-file/return-data/PRM.EDTOUT.DGMPRST...,23,63,"Index(['customerInput_Uniqueid', 'customerInpu..."
PRM.EDTOUT.DGMPRSTG.P467634.20190331-ACC2-P001.CSV,data/match-file/return-data/PRM.EDTOUT.DGMPRST...,1846,196990,"Index(['permId', 'creditAsOfDate_creditAsOfDat..."


## Identify return application file

In [17]:
%%time

return_files_df['application_flag'] = return_files_df['columns_ix'].apply(
    lambda x: x.intersection(id_lt + join_lt).shape[0] == len(id_lt + join_lt)
)

return_files_df['application_flag']

CPU times: user 7.02 ms, sys: 0 ns, total: 7.02 ms
Wall time: 8.52 ms


PRM.EDTOUT.DGMPRSTG.1273669.PERF.CSV                  False
PRM.EDTOUT.DGMPRSTG.File2_SCRAM.csv                   False
PRM.EDTOUT.DGMPRSTG.File3_SCRAM.csv                   False
PRM.EDTOUT.DGMPRSTG.File4_SCRAM.csv                   False
PRM.EDTOUT.DGMPRSTG.File5_SCRAM.csv                    True
PRM.EDTOUT.DGMPRSTG.P467632.20180930-ACC2-P001.CSV    False
PRM.EDTOUT.DGMPRSTG.P467632.20180930.REJ2.P001.CSV    False
PRM.EDTOUT.DGMPRSTG.P467633.20181231-ACC2-P001.CSV    False
PRM.EDTOUT.DGMPRSTG.P467633.20181231.REJ2.P001.CSV    False
PRM.EDTOUT.DGMPRSTG.P467634.20190331-ACC2-P001.CSV    False
PRM.EDTOUT.DGMPRSTG.P467634.20190331.REJ2.P001.CSV    False
PRM.EDTOUT.DGMPRSTG.P467635.20190630-ACC2-P001.CSV    False
PRM.EDTOUT.DGMPRSTG.P467635.20190630.REJ2.P001.CSV    False
PRM.EDTOUT.DGMPRSTG.P467636.20190930-ACC2-P001.CSV    False
PRM.EDTOUT.DGMPRSTG.P467636.20190930.REJ2.P001.CSV    False
Name: application_flag, dtype: bool

## Read in return application data

In [18]:
%%time

return_application_key_sr = return_files_df.query('application_flag')['key_sr'].squeeze()
print(f'Return application key: {return_application_key_sr}\n')

return_application_df = pd.read_csv(
    f's3://{bucket_sr}/{return_application_key_sr}',
    usecols=id_lt + join_lt, 
    **read_csv_dt
)

return_application_df.info()

Return application key: data/match-file/return-data/PRM.EDTOUT.DGMPRSTG.File5_SCRAM.csv



<timed exec>:4: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 972297 entries, 0 to 972296
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   UniqueID          972297 non-null  int64  
 1   bigAccountId      972297 non-null  int64  
 2   bigDebtorId       972297 non-null  int64  
 3   bitDebtor         972297 non-null  int64  
 4   strZipCode        971870 non-null  object 
 5   ApplicationDate   972297 non-null  int64  
 6   dtmStampCreation  972297 non-null  object 
 7   dtmDeclined       737876 non-null  object 
 8   bitNew            621404 non-null  float64
dtypes: float64(1), int64(5), object(3)
memory usage: 66.8+ MB
CPU times: user 4.77 s, sys: 463 ms, total: 5.23 s
Wall time: 16.5 s


## Handle mixed types in zip

In [19]:
%%time

print('Before:')
print(get_top_3_values_grouped_by_length(return_application_df['strZipCode']))

return_application_df['strZipCode'] = return_application_df['strZipCode'].apply(standardize_strings)
    
print('\nAfter:')
print(get_top_3_values_grouped_by_length(return_application_df['strZipCode']))

Before:
strZipCode
4.0                                   [6051]
5.0                    [20019, 60620, 20020]
6.0                         [466116, 611115]
10.0    [95901-8242, 76705-2376, 75034-4050]
NaN              [20019.0, 20020.0, 60620.0]
Name: strZipCode, dtype: object

After:
strZipCode
1                                      [0]
3                          [nan, 802, 791]
4                       [8021, 8081, 8618]
5                    [20019, 20020, 60620]
6                 [466116, 322226, 611115]
10    [95901-8242, 76705-2376, 75034-4050]
Name: strZipCode, dtype: object
CPU times: user 1.41 s, sys: 35.5 ms, total: 1.45 s
Wall time: 1.45 s


## Create join key

In [21]:
%%time

return_application_df[join_sr] = (
    return_application_df
    .loc[:, join_lt]
    .astype(str)
    .apply(lambda x: '_'.join(x), axis=1)
)

return_application_df[join_sr]

CPU times: user 6.9 s, sys: 110 ms, total: 7.01 s
Wall time: 7.03 s


0         2018-10-01T19:13:56Z_0_39110_2018-10-01T19:14:...
1         2018-10-01T19:05:19Z_0_61610_2018-10-01T19:05:...
2         2018-10-01T23:57:17Z_1_23464_2018-10-01T23:57:...
3         2018-10-01T19:18:20Z_1_60644_2018-10-01T19:20:...
4         2018-10-02T18:21:32Z_1_78412_2018-10-02T18:21:...
                                ...                        
972292    2019-04-02T00:32:35Z_1_36695_2019-04-02T16:27:...
972293    2019-04-10T23:43:43Z_1_92336_2019-04-10T23:43:...
972294    2019-04-10T20:42:38Z_1_36619_2019-04-10T20:43:...
972295          2019-04-10T23:08:58Z_1_nan_nan_0.0_20190410
972296    2019-04-11T00:28:07Z_1_97317_2019-04-11T00:28:...
Name: JoinKey, Length: 972297, dtype: object

## Drop duplicates

In [22]:
%%time

shape_te = return_application_df.shape

print(f'Shape before: {shape_te}')

return_application_df = return_application_df.loc[:, id_lt + [join_sr]].drop_duplicates(join_sr)

shape_te2 = return_application_df.shape

print(f'Shape after: {shape_te2}')
print(f'Difference: {np.array(shape_te2) - np.array(shape_te)}')

Shape before: (972297, 10)
Shape after: (972292, 4)
Difference: [-5 -6]
CPU times: user 482 ms, sys: 0 ns, total: 482 ms
Wall time: 481 ms


# Join sent and return application data

## Perform join

The `assert` statement guarantees that we do not gain or lose rows as a result of the join.

In [28]:
%%time

merged_df = sent_application_df.merge(return_application_df, how='inner', on=join_sr, suffixes=('', '_TU'))

assert sent_application_df.shape[0] == merged_df.shape[0]

merged_df

CPU times: user 861 ms, sys: 0 ns, total: 861 ms
Wall time: 1.08 s


,UniqueID,bigAccountId,bigDebtorId,JoinKey,UniqueID_TU,bigAccountId_TU,bigDebtorId_TU
0,390847750287411,3908477,5028741,2018-10-01T19:13:56Z_1_39110_2018-10-01T19:14:...,524304540300867,5243045,7363319
1,390847750287420,3908477,5028742,2018-10-01T19:13:56Z_0_39110_2018-10-01T19:14:...,524304540300876,5243045,7363310
2,390933050297991,3909330,5029799,2018-10-01T23:35:46Z_1_63136_2018-10-01T23:35:...,524490840310347,5244908,7364367
3,390933050298000,3909330,5029800,2018-10-01T23:35:46Z_0_63136_2018-10-01T23:35:...,524490840311456,5244908,7364478
4,390844350286981,3908443,5028698,2018-10-01T19:05:19Z_1_61610_2018-10-01T19:05:...,524301140309337,5243011,7363266
...,...,...,...,...,...,...,...
972287,431584855214810,4315848,5521481,2019-04-11T00:37:58Z_0_61364_2019-04-11T00:38:...,665041645337266,6650416,7866059
972288,431564355212291,4315643,5521229,2019-04-10T23:00:23Z_1_87401_nan_0.0_20190410,665021145335647,6650211,7866897
972289,431549455210341,4315494,5521034,2019-04-10T22:01:27Z_1_78574_2019-04-10T22:01:...,665006245333797,6650062,7866602
972290,431577355213891,4315773,5521389,2019-04-10T23:48:10Z_1_98466_2019-04-10T23:48:...,665034145336247,6650341,7866957


## Write to S3

In [32]:
%%time

merged_df.to_csv(f's3://{bucket_sr}/{prefix_sr}/match_file.zip')

CPU times: user 13.4 s, sys: 95.3 ms, total: 13.5 s
Wall time: 14.8 s
